In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/part-00000-tid-2816867743098933561-690107d0-3758-4beb-9e15-64c50c15041e-566-1-c000.csv")
test_df = pd.read_csv("/content/part-00000-tid-4519722107359529853-ac4c2059-cb7b-4268-a3b3-be3bdd788e6c-567-1-c000.csv")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining data:")
display(train_df.head())

print("\nTesting data:")
display(test_df.head())

Training shape: (24877, 3)
Testing shape: (25000, 3)

Training data:


,clean_text,label,text_length
0,"Caught this on IFC yesterday, and can't believ...",0,939
1,"Because IT IS, that's why! This is the same je...",0,338
2,SHALLOW GRAVE begins with either a tribute or ...,0,2267
3,"After a day at work, I sat down to relax and t...",0,589
4,Any time a movie is so myopic in its desire to...,0,1042



Testing data:


,clean_text,label,text_length
0,I love sci-fi and am willing to put up with a ...,0,1386
1,"Worth the entertainment value of a rental, esp...",0,1251
2,its a totally average film with a few semi-alr...,0,704
3,STAR RATING: ***** Saturday Night **** Friday ...,0,2039
4,"First off let me say, If you haven't enjoyed a...",0,672


In [ ]:
# Check data types
print("DATA TYPES")
print(train_df.dtypes)

# Check missing values
print("\nMISSING VALUES")
print("Training:\n", train_df.isnull().sum())
print("\nTesting:\n", test_df.isnull().sum())

# Check class distribution
print("\nTRAINING LABEL DISTRIBUTION")
print(train_df["label"].value_counts().sort_index())

print("\nTESTING LABEL DISTRIBUTION")
print(test_df["label"].value_counts().sort_index())

# Check empty reviews
print("\nEMPTY REVIEWS")
print("Training:", (train_df["clean_text"].str.strip() == "").sum())
print("Testing:", (test_df["clean_text"].str.strip() == "").sum())

# Check unexpected labels
print("\nUNIQUE LABELS")
print("Training:", sorted(train_df["label"].unique()))
print("Testing:", sorted(test_df["label"].unique()))

DATA TYPES
clean_text     object
label           int64
text_length     int64
dtype: object

MISSING VALUES
Training:
 clean_text     0
label          0
text_length    0
dtype: int64

Testing:
 clean_text     0
label          0
text_length    0
dtype: int64

TRAINING LABEL DISTRIBUTION
label
0    12405
1    12472
Name: count, dtype: int64

TESTING LABEL DISTRIBUTION
label
0    12500
1    12500
Name: count, dtype: int64

EMPTY REVIEWS
Training: 0
Testing: 0

UNIQUE LABELS
Training: [np.int64(0), np.int64(1)]
Testing: [np.int64(0), np.int64(1)]


In [ ]:
# Step 2 — Load the DistilBERT tokenizer

!pip install -q transformers

from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Model:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum model positions:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Model: distilbert/distilbert-base-uncased
Vocabulary size: 30522
Maximum model positions: 512


In [ ]:
# Step 3 — Understand tokenization with a real review

sample_text = train_df["clean_text"].iloc[0]

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original review:")
print(sample_text[:500])

print("\nTokens:")
print(tokens[:50])

print("\nToken IDs:")
print(token_ids[:50])

print("\nTotal tokens:", len(tokens))

Original review:
Caught this on IFC yesterday, and can't believe the positive reviews! Am I the only one who thought these "ladies" were anything but? Kate tells Jed she could get fired because she's supposed to be a pillar of the community, but puts out for him! Then they suddenly decide they're in love? And she's SO devastated over his death, she doesn't go to his funeral, much less, tell his family the "good news"! By the way, how did an American get to be the headmistress of a very proper British school? Jan

Tokens:
['caught', 'this', 'on', 'if', '##c', 'yesterday', ',', 'and', 'can', "'", 't', 'believe', 'the', 'positive', 'reviews', '!', 'am', 'i', 'the', 'only', 'one', 'who', 'thought', 'these', '"', 'ladies', '"', 'were', 'anything', 'but', '?', 'kate', 'tells', 'jed', 'she', 'could', 'get', 'fired', 'because', 'she', "'", 's', 'supposed', 'to', 'be', 'a', 'pillar', 'of', 'the', 'community']

Token IDs:
[3236, 2023, 2006, 2065, 2278, 7483, 1010, 1998, 2064, 1005, 1056, 2903, 1

In [ ]:
# Step 4 — Explore special tokens, padding and truncation

sample_text = train_df["clean_text"].iloc[0]

encoded = tokenizer(
    sample_text,
    max_length=256,
    padding="max_length",
    truncation=True,
    return_attention_mask=True
)

print("Input IDs:")
print(encoded["input_ids"][:30])

print("\nAttention Mask:")
print(encoded["attention_mask"][:30])

print("\nDecoded tokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][:30]))

print("\nFinal sequence length:", len(encoded["input_ids"]))
print("Actual tokens retained:", sum(encoded["attention_mask"]))

Input IDs:
[101, 3236, 2023, 2006, 2065, 2278, 7483, 1010, 1998, 2064, 1005, 1056, 2903, 1996, 3893, 4391, 999, 2572, 1045, 1996, 2069, 2028, 2040, 2245, 2122, 1000, 6456, 1000, 2020, 2505]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded tokens:
['[CLS]', 'caught', 'this', 'on', 'if', '##c', 'yesterday', ',', 'and', 'can', "'", 't', 'believe', 'the', 'positive', 'reviews', '!', 'am', 'i', 'the', 'only', 'one', 'who', 'thought', 'these', '"', 'ladies', '"', 'were', 'anything']

Final sequence length: 256
Actual tokens retained: 233


In [ ]:
# 2. Next step — Inspect the padding positions

print("Last 30 input IDs:")
print(encoded["input_ids"][-30:])

print("\nLast 30 attention mask values:")
print(encoded["attention_mask"][-30:])

print("\nPadding token ID:", tokenizer.pad_token_id)
print("Padding token:", tokenizer.pad_token)

print(
    "\nNumber of padding tokens:",
    encoded["input_ids"].count(tokenizer.pad_token_id)
)

Last 30 input IDs:
[2369, 2023, 1000, 3185, 1000, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Last 30 attention mask values:
[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Padding token ID: 0
Padding token: [PAD]

Number of padding tokens: 23


In [ ]:
# Step 5 — Test truncation

# Find the longest review in the training dataset
longest_index = train_df["text_length"].idxmax()

long_review = train_df.loc[longest_index, "clean_text"]

# Tokenize without truncation
full_tokens = tokenizer(
    long_review,
    truncation=False
)

# Tokenize with truncation
truncated_tokens = tokenizer(
    long_review,
    max_length=256,
    truncation=True
)

print("Original character length:",
      len(long_review))

print("Token count before truncation:",
      len(full_tokens["input_ids"]))

print("Token count after truncation:",
      len(truncated_tokens["input_ids"]))

print("Maximum sequence length:", 256)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3047 > 512). Running this sequence through the model will result in indexing errors


Original character length: 13602
Token count before truncation: 3047
Token count after truncation: 256
Maximum sequence length: 256


In [ ]:
# 3. Next step — Batch tokenization

# Step 6 — Tokenize the training dataset
MAX_LENGTH = 256

train_encodings = tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

print("Training tokenization completed!")

print("Number of reviews:", len(train_encodings["input_ids"]))

print("First review token count:",
      len(train_encodings["input_ids"][0]))

print("Longest tokenized review:",
      max(len(ids) for ids in train_encodings["input_ids"]))

Training tokenization completed!
Number of reviews: 24877
First review token count: 233
Longest tokenized review: 256


In [ ]:
# Step 7 — Tokenize the test dataset

test_encodings = tokenizer(
    test_df["clean_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

print("Testing tokenization completed!")

print("Number of reviews:", len(test_encodings["input_ids"]))

print("First review token count:",
      len(test_encodings["input_ids"][0]))

print("Longest tokenized review:",
      max(len(ids) for ids in test_encodings["input_ids"]))

Testing tokenization completed!
Number of reviews: 25000
First review token count: 256
Longest tokenized review: 256


In [ ]:
# Step 8 — Understand the PyTorch Dataset

import torch
from torch.utils.data import Dataset

class IMDbDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item


# Create training and testing datasets
train_dataset = IMDbDataset(
    train_encodings,
    train_df["label"].tolist()
)

test_dataset = IMDbDataset(
    test_encodings,
    test_df["label"].tolist()
)

print("PyTorch datasets created successfully!")

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

PyTorch datasets created successfully!
Training samples: 24877
Testing samples: 25000


In [ ]:
# Step 9 — Inspect one PyTorch sample

sample = train_dataset[0]

print("Available keys:", sample.keys())

print("\nInput IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)

print("\nLabel:", sample["labels"].item())

print("\nFirst 20 input IDs:")
print(sample["input_ids"][:20])

print("\nFirst 20 attention mask values:")
print(sample["attention_mask"][:20])

Available keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

Input IDs shape: torch.Size([233])
Attention mask shape: torch.Size([233])

Label: 0

First 20 input IDs:
tensor([ 101, 3236, 2023, 2006, 2065, 2278, 7483, 1010, 1998, 2064, 1005, 1056,
        2903, 1996, 3893, 4391,  999, 2572, 1045, 1996])

First 20 attention mask values:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [ ]:
# Step 10 — Configure dynamic padding

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

print("Dynamic padding collator created!")

Dynamic padding collator created!


In [ ]:
# Step 11 — Create PyTorch DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("DataLoaders created successfully!")

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully!
Training batches: 3110
Testing batches: 3125


In [ ]:
# Step 12 — Inspect one actual batch

# Get one batch from the training DataLoader
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())

print("\nInput IDs shape:")
print(batch["input_ids"].shape)

print("\nAttention mask shape:")
print(batch["attention_mask"].shape)

print("\nLabels shape:")
print(batch["labels"].shape)

print("\nLabels in this batch:")
print(batch["labels"])

print("\nActual sequence length in this batch:")
print(batch["attention_mask"].sum(dim=1))

Batch keys: KeysView({'input_ids': tensor([[ 101, 2023, 2143,  ..., 1005, 1055,  102],
        [ 101, 2023, 3185,  ...,    0,    0,    0],
        [ 101, 1045, 1010,  ...,    0,    0,    0],
        ...,
        [ 101, 1045, 2031,  ...,    0,    0,    0],
        [ 101, 2332, 1005,  ..., 2153, 1010,  102],
        [ 101, 1045, 2467,  ..., 2001, 1011,  102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([0, 0, 0, 0, 1, 0, 1, 1])})

Input IDs shape:
torch.Size([8, 256])

Attention mask shape:
torch.Size([8, 256])

Labels shape:
torch.Size([8])

Labe

In [ ]:
# 3. Next step — Load the DistilBERT classification model

from transformers import (
    AutoModelForSequenceClassification
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("DistilBERT model loaded successfully!")

print("\nNumber of labels:", model.config.num_labels)
print("Hidden size:", model.config.dim)
print("Transformer layers:", model.config.n_layers)
print("Attention heads:", model.config.n_heads)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!

Number of labels: 2
Hidden size: 768
Transformer layers: 6
Attention heads: 12


In [ ]:
# 3. Next step — Check available hardware

import torch

print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GB"
    )
else:
    print("GPU not available. Currently using CPU.")

PyTorch version: 2.11.0+cu130
GPU available: Tesla T4
CUDA version: 13.0
GPU memory: 14.56 GB


In [ ]:
# 2. Step 13 — Move the model to GPU

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

Model device: cuda:0
GPU: Tesla T4
Model parameters: 66955010


In [ ]:
# Step 14 — Configure the optimizer

from torch.optim import AdamW

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer configured successfully!")
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Optimizer configured successfully!
Optimizer: AdamW
Learning rate: 2e-05
Weight decay: 0.01


In [ ]:
# Step 15 — Configure the learning-rate scheduler

from transformers import get_linear_schedule_with_warmup
import math

EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 2

total_training_steps = (
    math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    * EPOCHS
)

warmup_steps = int(0.1 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Scheduler configured successfully!")
print("Training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Epochs:", EPOCHS)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

Scheduler configured successfully!
Training steps: 3110
Warmup steps: 311
Epochs: 2
Gradient accumulation: 2


In [ ]:
# Step 16 — Configure mixed precision training
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

print("Mixed precision configuration ready!")
print("FP16 supported:", torch.cuda.is_available())
print("Gradient scaler initialized:", scaler.is_enabled())

Mixed precision configuration ready!
FP16 supported: True
Gradient scaler initialized: True


/tmp/ipykernel_1365/2082474343.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# Step 16A — Update GradScaler

from torch.amp import GradScaler, autocast

scaler = GradScaler("cuda")

print("GradScaler initialized:", scaler.is_enabled())
print("Mixed precision:", "FP16")

GradScaler initialized: True
Mixed precision: FP16


In [ ]:
# Step 17 — Prepare the training loop
# Step 17A — Initialize training configuration

import torch
import numpy as np

EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 2

# Clear unused GPU memory
torch.cuda.empty_cache()

# Initialize loss tracking
training_history = {
    "epoch": [],
    "training_loss": []
}

# Initialize gradients
optimizer.zero_grad(set_to_none=True)

print("Training configuration initialized!")
print("Epochs:", EPOCHS)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / (1024**3), 2), "GB")

Training configuration initialized!
Epochs: 2
Gradient accumulation: 2
GPU memory allocated: 0.25 GB


In [ ]:
# Step 18 — Build the training loop

from tqdm.auto import tqdm
import os

checkpoint_dir = "/content/imdb_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in enumerate(progress_bar):

        # Keep only inputs required by DistilBERT
        batch = {
            key: value.to(device)
            for key, value in batch.items()
            if key in ["input_ids", "attention_mask", "labels"]
        }

        with autocast("cuda", dtype=torch.float16):

            outputs = model(**batch)
            loss = outputs.loss

            # Scale loss for gradient accumulation
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        # Update weights after every two batches
        if (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        ):

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        progress_bar.set_postfix(
            loss=f"{total_loss / (step + 1):.4f}"
        )

    epoch_loss = total_loss / len(train_loader)

    training_history["epoch"].append(epoch + 1)
    training_history["training_loss"].append(epoch_loss)

    print(f"\nEpoch {epoch + 1} Training Loss: {epoch_loss:.4f}")

    # Save checkpoint
    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"epoch_{epoch + 1}"
    )

    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)

    print("Checkpoint saved:", checkpoint_path)

Epoch 1/2:   0%|          | 0/3110 [00:00<?, ?it/s]


Epoch 1 Training Loss: 0.3014


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: /content/imdb_checkpoints/epoch_1


Epoch 2/2:   0%|          | 0/3110 [00:00<?, ?it/s]


Epoch 2 Training Loss: 0.1534


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: /content/imdb_checkpoints/epoch_2


In [ ]:
# Step 19 — Evaluate the model on the test dataset

import torch
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_predictions = []
all_labels = []
total_test_loss = 0

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Evaluating"):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
            if key in ["input_ids", "attention_mask", "labels"]
        }

        with autocast("cuda", dtype=torch.float16):
            outputs = model(**batch)

        total_test_loss += outputs.loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print("TEST LOSS:", round(total_test_loss / len(test_loader), 4))
print("ACCURACY:", round(accuracy, 4))
print("PRECISION:", round(precision, 4))
print("RECALL:", round(recall, 4))
print("F1 SCORE:", round(f1, 4))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Evaluating:   0%|          | 0/3125 [00:00<?, ?it/s]

TEST LOSS: 0.2213
ACCURACY: 0.9169
PRECISION: 0.9072
RECALL: 0.9287
F1 SCORE: 0.9179

Classification Report:
              precision    recall  f1-score   support

    Negative       0.93      0.91      0.92     12500
    Positive       0.91      0.93      0.92     12500

    accuracy                           0.92     25000
   macro avg       0.92      0.92      0.92     25000
weighted avg       0.92      0.92      0.92     25000


Confusion Matrix:
[[11313  1187]
 [  891 11609]]


In [ ]:
# Step 20 — Save the final model

final_model_path = "/content/imdb_distilbert_final"

model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("Final model saved successfully!")
print("Location:", final_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved successfully!
Location: /content/imdb_distilbert_final


In [ ]:
# Step 1 — Create the error analysis DataFrame

import pandas as pd

error_df = test_df[["clean_text", "label"]].copy()

error_df["predicted_label"] = all_predictions

error_df["actual_sentiment"] = error_df["label"].map({
    0: "Negative",
    1: "Positive"
})

error_df["predicted_sentiment"] = error_df["predicted_label"].map({
    0: "Negative",
    1: "Positive"
})

error_df["correct_prediction"] = (
    error_df["label"] == error_df["predicted_label"]
)

misclassified_df = error_df[
    error_df["correct_prediction"] == False
].copy()

print("Total test reviews:", len(error_df))
print("Correct predictions:", error_df["correct_prediction"].sum())
print("Misclassified reviews:", len(misclassified_df))
print("\nMisclassification distribution:")
print(
    misclassified_df.groupby(
        ["actual_sentiment", "predicted_sentiment"]
    ).size()
)

Total test reviews: 25000
Correct predictions: 22922
Misclassified reviews: 2078

Misclassification distribution:
actual_sentiment  predicted_sentiment
Negative          Positive               1187
Positive          Negative                891
dtype: int64


In [ ]:
# Step 2 — Display 10 misclassified reviews

pd.set_option("display.max_colwidth", 500)

sample_errors = misclassified_df[
    [
        "actual_sentiment",
        "predicted_sentiment",
        "clean_text"
    ]
].sample(
    n=10,
    random_state=42
)

display(sample_errors)

,actual_sentiment,predicted_sentiment,clean_text
6258,Negative,Positive,"The story and the characters really REALLY needed work. The world idea is kind of neat, but no one bothered to develop any of it either through exposition, or through the plot. Despite the cheesy notes at the beginning of the film, it makes sense that you wouldn't use exposition, since no one is new to this world. And yet, when Matt Owens (Bill Paxton) and Byron (Bob Peck) stray off-course and get lost, and get introduced to the wind-worshipers and the fat, lazy, rich people in the museum, w..."
17080,Positive,Negative,"Watching Stranger Than Fiction director Marc Forster's The Kite Runner is the cinematic equivalent of eating your vegetables because this art-house epic rated PG-13 is good for your movie-going diet. No, this isn't the kind of movie that I like to slouch on the couch and eyeball at the end of a tough day. The Kite Runner isn't your typical mainstream movie designed to entertain you and make you forget about your troubles. First, no celebrity stars appear in it. Second, nothing is cut and dri..."
14753,Positive,Negative,I saw this movie in a theater while on vacation in Pablo CO. I had just quit my biomedical engineering job at a hospital. I consider the script to be a exaggeration of the real type of stuff that goes on in hospitals. The idiots that put it down on production value don't get the point and probably have never been hospitalized. And never worked in one for sure. Billy Jack (same era) was very poorly produced but had a significant social comment and was a very good movie with a real social me...
16153,Positive,Negative,"The animation quality here is decidedly below par IMO, regardless of the age of the OAV. The plot itself makes sense, but the characters don't. This might sound picky, but considering that the city of Olympus is almost exclusively populated by living machines, would YOU feel comfortable there as a human? Would any machine ever emulate human arrogance, short-sightedness and greed quite so effectively? I doubt it. There are some pointers towards this in the movie, but only when the team can b..."
7893,Negative,Positive,"The Kissing Bandit was the third and final film that Frank Sinatra and Kathryn Grayson co-starred at MGM with. The first two were Anchors Aweigh and It Happened in Brooklyn. And in both Sinatra wooed and lost Grayson. I guess the third time's the charm. For romance maybe, but definitely not for screen image. Sinatra in his forty's films once again plays the nice little schnook only this time in toreador pants. Poaching on Tyrone Power's territory laid out in The Mark of Zorro, Sinatra plays..."
3880,Negative,Positive,"I preface by stating I am a big fan of JJL and NOT one of Patrick. Therefore I watched this to see her performance and of course, it was excellent. I do not feel the director was adequate for the film as several very bad choices were made re: shot angles, blocking, etc. If the director was trying to give it a ""realistic"" feel, they failed and lost some good performances because of it. Nearly always felt that the camera was way too static, too far from intense facial reactions -- and so many ..."
22322,Positive,Negative,"If you love the book, as I do, stop watching the video after Jean and Joe meet in Australia. Up to that point it is a fairly faithful rendition of the book, and the visuals are great. 10 out of 10 to that point and I've enjoyed it many times. After that, the story is seriously rearranged and revised in ways that really destroy the key part of the book, i.e., how Jean creates a town like Alice (Springs). In the early part, the major change is to make Strachan a 40-something bachelor instead o..."
3277,Negative,Positive,"This movie beats everything out there. Well, depends on what you are looking for... it could be a 10 or a 1 on the scale. This movie is in a complete league of its own.. I don't think any movie could possibly come close to it. I am not sure what the di

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./imdb_distilbert_final"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
import torch
import torch.nn.functional as F

def predict_sentiment(review):
    model.eval()

    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        with autocast("cuda", dtype=torch.float16):
            outputs = model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)

    predicted_class = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_class].item() * 100

    sentiment = "Positive" if predicted_class == 1 else "Negative"

    return {
        "sentiment": sentiment,
        "confidence": round(confidence, 2)
    }

In [ ]:
print("Model device:", next(model.parameters()).device)
print("Configured device:", device)

Model device: cpu
Configured device: cuda


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

print("Model device:", next(model.parameters()).device)
print("Configured device:", device)

Model device: cuda:0
Configured device: cuda


In [ ]:
review = """
This movie was absolutely fantastic!
The story was engaging, the acting was brilliant,
and the direction was outstanding.
I thoroughly enjoyed every minute of it.
"""

result = predict_sentiment(review)

print("Movie Review:")
print(review)

print("\nPrediction:", result["sentiment"])
print("Confidence:", result["confidence"], "%")

Movie Review:

This movie was absolutely fantastic!
The story was engaging, the acting was brilliant,
and the direction was outstanding.
I thoroughly enjoyed every minute of it.


Prediction: Positive
Confidence: 99.66 %


In [ ]:
# Step 3 — Test a negative review

negative_review = """
This movie was a complete disappointment.
The story was boring, the acting was terrible,
and the direction was extremely poor.
I regret wasting my time watching it.
"""

negative_result = predict_sentiment(negative_review)

print("Movie Review:")
print(negative_review)

print("\nPrediction:", negative_result["sentiment"])
print("Confidence:", negative_result["confidence"], "%")

Movie Review:

This movie was a complete disappointment.
The story was boring, the acting was terrible,
and the direction was extremely poor.
I regret wasting my time watching it.


Prediction: Negative
Confidence: 99.56 %


In [ ]:
!pip install streamlit
import torch
import transformers
import streamlit

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Streamlit:", streamlit.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 10.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Streamlit: 1.66.1
